# 02 — Simulate SNe Ia in the LSST DP2 Deep Drilling Fields with `skysurvey` using visits Vutler

**Goal.** Use the DP2 (Data Preview 2) visits table (`dp2_visits_table_with_iq.ecsv`) as the
observing-strategy input to [`skysurvey`](https://github.com/MickaelRigault/skysurvey)
(Rigault et al. 2026, [arXiv:2605.25840](https://arxiv.org/abs/2605.25840)) and simulate the
population of Type Ia supernovae that DP2 would have observed in its Deep Drilling Fields (DDF).

**Strategy.**
1. Read the DP2 visits table and auto-detect the columns skysurvey needs
   (`ra`, `dec`, `mjd`, `band`, `skynoise`, `gain`, `zp`).
2. If the table does not directly provide a 5-sigma limiting magnitude (`m5`) and/or a
   `skynoise`, derive them from sky brightness / seeing / airmass / exptime using the standard
   LSST SNR formalism documented in
   [SMTN-002](https://smtn-002.lsst.io) (Jones 2025), the same approach `skysurvey.LSST.from_opsim`
   uses internally for OpSim databases.
3. Build a `skysurvey.LSST` survey object (`Survey.from_pointings`-based, healpix-matched) from
   this observing log.
4. Draw a population of SNe Ia (`skysurvey.SNeIa.from_draw`) over the sky area actually covered by
   the DDF pointings and over the DP2 time range.
5. Match targets with the survey (`skysurvey.DataSet.from_targets_and_survey`) to obtain the
   simulated, DP2-sampled SN Ia light curves.
6. Save data products and diagnostic figures.

**Important note on real column names.** DP2 visit-table columns can vary depending on how the
table was produced (Butler `visitTable`/`ccdVisitTable` export, custom query, etc.). Section 2
below prints the columns actually found in your file and the mapping that was inferred. **Check
that printout** the first time you run this notebook, and edit the `CANDIDATES` dictionary in
Section 2 if a column was not auto-detected correctly.

**Conventions.** English-only code/comments; kernel `conda_py313`; outputs go to
`data_out_simSNIaDDF/` and `figs_out_simSNIaDDF/`; figures are saved as PDF+PNG.


- **author :** Sylvie Dagoret-Campagne
- **affiliation :** IJCLab/IN2P3/CNRS
- **creation date :** 2026-07-23
- **context :** DESC sprint  on TD ans SN

## 1. Imports and configuration

In [4]:
# Standard library
import os
import warnings
from pathlib import Path

# Scientific stack
import numpy as np
import pandas as pd
from astropy.table import Table
from astropy.time import Time
from astropy.io import ascii as astropy_ascii

import matplotlib.pyplot as plt

# skysurvey
import skysurvey
from skysurvey.tools.utils import get_skynoise_from_maglimit

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams["figure.dpi"] = 100


        Use pytest instead. [astropy.tests.runner]
        Use pytest instead. [astropy.utils.decorators]


In [15]:
!ls data_lsstvisists_in

constDbVisitTable-2025041500043-2026053000874_N117815_WithTracts.parquet
visitTable-2025041500138-2026053000760_N83426_WithTracts.parquet


In [29]:
# ----------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------
NB_TAG = "02_simSNIaDDF"

DATA_IN_DIR  = Path("data_lsstvisists_in")
DATA_OUT_DIR = Path(f"data_out_{NB_TAG}")
FIGS_OUT_DIR = Path(f"figs_out_{NB_TAG}")
DATA_OUT_DIR.mkdir(exist_ok=True)
FIGS_OUT_DIR.mkdir(exist_ok=True)

#VISITS_FILE = DATA_IN_DIR / "visitTable-2025041500138-2026053000760_N83426_WithTracts.parquet"
VISITS_FILE = DATA_IN_DIR / "constDbVisitTable-2025041500043-2026053000874_N117815_WithTracts.parquet"

# skysurvey internal photometric convention: a single, arbitrary zeropoint used
# consistently for zp and skynoise (this is what skysurvey.LSST.from_opsim uses too;
# it does NOT need to match the "real" instrumental zeropoint of the visit).
SKYSURVEY_ZP = 31.4
GAIN = 1.0

# SN Ia population parameters
ZMIN = 0.001
ZMAX = 0.6
RANDOM_SEED = 42

print(f"Visits file: {VISITS_FILE}  (exists: {VISITS_FILE.exists()})")


Visits file: data_lsstvisists_in/constDbVisitTable-2025041500043-2026053000874_N117815_WithTracts.parquet  (exists: True)


## 2. Load the DP2 visits table and detect columns

We read the `.ecsv` file with `astropy.table.Table` (which understands the ECSV metadata header),
convert to a `pandas.DataFrame`, and print the columns so the auto-detection below can be checked
and corrected if needed.

In [30]:
#visits_tbl = Table.read(VISITS_FILE, format="ascii.ecsv")
#visits_tbl = astropy_ascii.read(VISITS_FILE)
#visits = visits_tbl.to_pandas()
visits  = pd.read_parquet(VISITS_FILE)

print(f"{len(visits)} visits read.")
print("\nColumns found in the visits table:")
for c in visits.columns:
    print(f"  - {c:30s} dtype={visits[c].dtype}")

visits.head()


117815 visits read.

Columns found in the visits table:
  - visit_id                       dtype=int64
  - exposure_name                  dtype=str
  - controller                     dtype=str
  - day_obs                        dtype=int64
  - seq_num                        dtype=int64
  - physical_filter                dtype=str
  - band                           dtype=str
  - s_ra                           dtype=float64
  - s_dec                          dtype=float64
  - sky_rotation                   dtype=float64
  - azimuth_start                  dtype=float64
  - azimuth_end                    dtype=float64
  - azimuth                        dtype=float64
  - altitude_start                 dtype=float64
  - altitude_end                   dtype=float64
  - altitude                       dtype=float64
  - zenith_distance_start          dtype=float64
  - zenith_distance_end            dtype=float64
  - zenith_distance                dtype=float64
  - airmass                        

,visit_id,exposure_name,controller,day_obs,seq_num,physical_filter,band,s_ra,s_dec,sky_rotation,...,scheduler_note,s_region,can_see_sky,pgs_region,tract,patch,patch_str,tract_bbox,tract_ra_corners,tract_dec_corners
0,2025102300030,MC_O_20251023_000030,O,20251023,30,r_57,r,3.049752,-26.004924,163.744237,...,closed_loop_22dof_trunc12,NaN,True,NaN,5250,56,"6,5","[0, 0, 29999, 29999]","[4.22159098068198, 2.3536892306332255, 2.36686...","[-26.863208457240876, -26.863208048146397, -25..."
1,2025102300031,MC_O_20251023_000031,O,20251023,31,r_57,r,3.049828,-26.004872,163.744147,...,closed_loop_22dof_trunc12,NaN,True,NaN,5250,56,"6,5","[0, 0, 29999, 29999]","[4.22159098068198, 2.3536892306332255, 2.36686...","[-26.863208457240876, -26.863208048146397, -25..."
2,2025102300032,MC_O_20251023_000032,O,20251023,32,r_57,r,3.499502,-26.004841,163.744147,...,closed_loop_22dof_trunc12,NaN,True,NaN,5250,53,"3,5","[0, 0, 29999, 29999]","[4.22159098068198, 2.3536892306332255, 2.36686...","[-26.863208457240876, -26.863208048146397, -25..."
3,2025102300033,MC_O_20251023_000033,O,20251023,33,r_57,r,3.499488,-26.004891,163.744232,...,closed_loop_22dof_trunc12,NaN,True,NaN,5250,53,"3,5","[0, 0, 29999, 29999]","[4.22159098068198, 2.3536892306332255, 2.36686...","[-26.863208457240876, -26.863208048146397, -25..."
4,2025102300034,MC_O_20251023_000034,O,20251023,34,r_57,r,3.951651,-26.004800,163.744108,...,closed_loop_22dof_trunc12,NaN,True,NaN,5250,51,"1,5","[0, 0, 29999, 29999]","[4.22159098068198, 2.3536892306332255, 2.36686...","[-26.863208457240876, -26.863208048146397, -25..."


In [22]:
visits["band"]

0         r
1         r
2         r
3         r
4         r
         ..
129309    y
129310    y
129311    y
129312    y
129313    y
Name: band, Length: 117815, dtype: str

In [31]:
# ----------------------------------------------------------------
# Column auto-detection
# ----------------------------------------------------------------
# Candidate names for each quantity skysurvey (or our m5 fallback) needs.
# Edit/extend these lists if your DP2 export uses different names.
CANDIDATES = {
    "visit_id":  ["visitId", "visit_id", "visit", "id"],
    "ra":        ["ra", "fieldRA", "fieldRa", "boresightRa", "pointing_ra","s_ra"],
    "dec":       ["dec", "decl", "fieldDec", "boresightDec", "pointing_dec","s_dec"],
    "mjd":       ["expMidptMJD", "observationStartMJD", "obsStartMJD", "mjd", "mjd_mid","exp_midpt_mjd"],
    "band":      ["band", "physical_filter", "filter", "filternamej"],
    "exptime":   ["expTime", "exptime", "visitExposureTime", "exposure_time","exp_time","expos"],
    "airmass":   ["airmass", "zenithDistance"],
    "seeing":    ["seeing", "FWHMeff", "fwhmEff", "psfSigma", "psfFwhm", "seeingFwhmEff","dimm_seeing"],
    "skybg":     ["skyBg", "skyBackground", "skyBrightness", "sky_mag"],
    "zeropoint": ["zeroPoint", "zp", "magzero", "zpNJy"],
    "maglim":    ["maglim", "m5", "fiveSigmaDepth", "fiveSigmaDepthMag"],
}


def find_col(df, candidates):
    # Return the first matching column name found in df, or None.
    for name in candidates:
        if name in df.columns:
            return name
    return None


detected = {key: find_col(visits, names) for key, names in CANDIDATES.items()}

print("Detected column mapping (None = not found in the table):")
for key, col in detected.items():
    print(f"  {key:12s} -> {col}")


Detected column mapping (None = not found in the table):
  visit_id     -> visit_id
  ra           -> s_ra
  dec          -> s_dec
  mjd          -> exp_midpt_mjd
  band         -> band
  exptime      -> exp_time
  airmass      -> airmass
  seeing       -> dimm_seeing
  skybg        -> None
  zeropoint    -> None
  maglim       -> None


## 3. Reference photometric constants (SMTN-002)

Fiducial LSST zeropoints (1s, gain=1), dark-sky brightness, effective seeing, and the
`Cm` / `dCm_infinity` / `k_atm` coefficients used to rescale the 5-sigma limiting magnitude
`m5` for arbitrary exposure time / sky brightness / seeing / airmass, taken from
[SMTN-002](https://smtn-002.lsst.io) (R. Lynne Jones, 2025). These are only used as a **fallback**
when the visits table does not directly give a usable `m5`/`maglim` column.

https://smtn-002.lsst.io/

In [32]:
SMTN002 = pd.DataFrame(
    {
        "zp_1s":       {"u": 26.52, "g": 28.51, "r": 28.36, "i": 28.17, "z": 27.78, "y": 26.82},
        "m_darksky":   {"u": 23.05, "g": 22.25, "r": 21.20, "i": 20.46, "z": 19.61, "y": 18.60},
        "fwhm_eff":    {"u": 0.92,  "g": 0.87,  "r": 0.83,  "i": 0.80,  "z": 0.78,  "y": 0.76},
        "m5_fiducial": {"u": 23.70, "g": 24.97, "r": 24.52, "i": 24.13, "z": 23.56, "y": 22.55},
        "Cm":          {"u": 22.97, "g": 24.58, "r": 24.60, "i": 24.54, "z": 24.37, "y": 23.84},
        "dCm_inf":     {"u": 0.54,  "g": 0.09,  "r": 0.04,  "i": 0.03,  "z": 0.02,  "y": 0.02},
        "k_atm":       {"u": 0.47,  "g": 0.21,  "r": 0.13,  "i": 0.10,  "z": 0.07,  "y": 0.17},
    }
)
SMTN002.index.name = "band"
SMTN002


,zp_1s,m_darksky,fwhm_eff,m5_fiducial,Cm,dCm_inf,k_atm
band,,,,,,,
u,26.52,23.05,0.92,23.70,22.97,0.54,0.47
g,28.51,22.25,0.87,24.97,24.58,0.09,0.21
r,28.36,21.20,0.83,24.52,24.60,0.04,0.13
i,28.17,20.46,0.80,24.13,24.54,0.03,0.10
z,27.78,19.61,0.78,23.56,24.37,0.02,0.07
y,26.82,18.60,0.76,22.55,23.84,0.02,0.17


In [33]:
def m5_from_conditions(band, sky_mag, fwhm_eff, exptime_s, airmass, exptime_ref_u=15.0, exptime_ref_other=30.0):
    '''5-sigma limiting magnitude from observing conditions (SMTN-002 formalism).

    Parameters
    ----------
    band : array-like of str
        Single-letter LSST band (u, g, r, i, z, y).
    sky_mag : array-like of float
        Sky brightness (mag/arcsec^2) for that visit.
    fwhm_eff : array-like of float
        Effective seeing FWHM (arcsec) for that visit.
    exptime_s : array-like of float
        Visit exposure time (s).
    airmass : array-like of float
        Visit airmass.

    Returns
    -------
    np.ndarray of m5 values (mag).
    '''
    band = np.asarray(band)
    sky_mag = np.asarray(sky_mag, dtype=float)
    fwhm_eff = np.asarray(fwhm_eff, dtype=float)
    exptime_s = np.asarray(exptime_s, dtype=float)
    airmass = np.asarray(airmass, dtype=float)

    Cm = SMTN002["Cm"].reindex(band).values
    dCm_inf = SMTN002["dCm_inf"].reindex(band).values
    k_atm = SMTN002["k_atm"].reindex(band).values
    m_darksky = SMTN002["m_darksky"].reindex(band).values

    # u band uses a 15s reference exposure for Tscale (read-noise limited regime),
    # other bands use 30s (sky-noise limited regime).
    exptime_ref = np.where(band == "u", exptime_ref_u, exptime_ref_other)

    Tscale = (exptime_s / exptime_ref) * 10.0 ** (-0.4 * (sky_mag - m_darksky))
    dCm = dCm_inf - 1.25 * np.log10(1.0 + (10.0 ** (0.8 * dCm_inf) - 1.0) / Tscale)

    m5 = (
        Cm + dCm
        + 0.50 * (sky_mag - 21.0)
        + 2.5 * np.log10(0.7 / fwhm_eff)
        + 1.25 * np.log10(exptime_s / 30.0)
        - k_atm * (airmass - 1.0)
    )
    return m5


## 4. Build the `skysurvey`-formatted observing log

We assemble the `pandas.DataFrame` skysurvey expects: `ra`, `dec`, `mjd`, `band`
(prefixed `lsst<letter>`, matching the built-in `sncosmo` bandpasses used by
`skysurvey.LSST`), `skynoise`, `gain`, `zp`.

`skynoise` is obtained from the 5-sigma limiting magnitude via
`skysurvey.tools.utils.get_skynoise_from_maglimit(m5, zp=SKYSURVEY_ZP)` — exactly the function
`skysurvey.LSST.from_opsim` itself calls, so a DP2-based survey and an OpSim-based survey remain
directly comparable.

In [34]:
d = detected  # shorthand

# --- band: normalize to a single lower-case letter (u,g,r,i,z,y) --------------
band_raw = visits[d["band"]].astype(str).str.lower()
# Strip common prefixes ("lsst_", "lsst-") DP2 exports sometimes carry, keep last letter token
band_letter = band_raw.str.replace("lsst", "", regex=False).str.strip("_- ").str[-1]
band_lsst = "lsst" + band_letter  # skysurvey / sncosmo bandpass names: lsstu, lsstg, ...

print("Band mapping check:")
print(pd.DataFrame({"raw": band_raw, "letter": band_letter, "lsst_band": band_lsst}).drop_duplicates())


Band mapping check:
      raw letter lsst_band
0       r      r     lsstr
102     y      y     lssty
164     i      i     lssti
432     z      z     lsstz
544     g      g     lsstg
11286   u      u     lsstu


### determination de m5

- $C_m : $ constante dépendant du filtre (instrument)
- $m_sky$ : brillance du ciel (mag/arcsec²)
- $\theta_{eff} :$ seeing effectif (PSF)
- $t_{exp} : $ temps de pose
- $k_m$ : coefficient d’extinction atmosphérique
- $X$ : airmass

$$ m_5 = C_m +0.50(m_{sky}−21)+2.5\log_{10}(0.7/θeff​)+1.25\log_{10}​(t_{exp}/30)−km​(X−1)$$

In [36]:
# --- m5 (5-sigma limiting magnitude) ------------------------------------------
if d["maglim"] is not None:
    print(f"Using native limiting-magnitude column '{d['maglim']}'.")
    m5 = visits[d["maglim"]].astype(float).values
else:
    print("No native m5/maglim column found -> deriving m5 from observing conditions (SMTN-002).")
    missing = [k for k in ("skybg", "seeing", "exptime") if d[k] is None]
    if missing:
        raise ValueError(
            f"Cannot derive m5: missing columns for {missing}. "
            "Edit the CANDIDATES dict in Section 2 to point to the right column names, "
            "or provide a native maglim/m5 column."
        )
    airmass_vals = (
        visits[d["airmass"]].astype(float).values
        if d["airmass"] is not None
        else np.ones(len(visits))  # fall back to airmass = 1 (zenith) if not available
    )
    if d["airmass"] is None:
        print("  (no airmass column found -> assuming airmass = 1 for all visits)")

    m5 = m5_from_conditions(
        band=band_letter.values,
        sky_mag=visits[d["skybg"]].astype(float).values,
        fwhm_eff=visits[d["seeing"]].astype(float).values,
        exptime_s=visits[d["exptime"]].astype(float).values,
        airmass=airmass_vals,
    )

print(f"m5 range: [{np.nanmin(m5):.2f}, {np.nanmax(m5):.2f}] mag")


No native m5/maglim column found -> deriving m5 from observing conditions (SMTN-002).


ValueError: Cannot derive m5: missing columns for ['skybg']. Edit the CANDIDATES dict in Section 2 to point to the right column names, or provide a native maglim/m5 column.

In [ ]:
# --- assemble the skysurvey observing-log DataFrame ----------------------------
survey_data = pd.DataFrame(
    {
        "ra":       visits[d["ra"]].astype(float).values,
        "dec":      visits[d["dec"]].astype(float).values,
        "mjd":      visits[d["mjd"]].astype(float).values,
        "band":     band_lsst.values,
        "skynoise": get_skynoise_from_maglimit(m5, zp=SKYSURVEY_ZP),
        "gain":     GAIN,
        "zp":       SKYSURVEY_ZP,
    }
)
if d["visit_id"] is not None:
    survey_data["visitId"] = visits[d["visit_id"]].values

print(f"{len(survey_data)} rows in the skysurvey observing log.")
survey_data.head()


In [ ]:
survey_data.to_parquet(DATA_OUT_DIR / "dp2_ddf_survey_data.parquet")
print(f"Saved -> {DATA_OUT_DIR / 'dp2_ddf_survey_data.parquet'}")


## 5. Build the `skysurvey.LSST` survey object

In [ ]:
survey = skysurvey.LSST(data=survey_data)

print(f"Bands covered: {sorted(survey.data['band'].unique())}")
tmin, tmax = survey.get_timerange() if hasattr(survey, "get_timerange") else (
    survey.data["mjd"].min(), survey.data["mjd"].max()
)
print(f"Start: {Time(tmin, format='mjd').iso}")
print(f"End:   {Time(tmax, format='mjd').iso}")
print(f"Duration: {(tmax - tmin) / 365.25:.2f} years")


In [ ]:
fig = survey.show_footprint(add_text=True)
fig.savefig(FIGS_OUT_DIR / "lsst_footprint.pdf")
fig.savefig(FIGS_OUT_DIR / "lsst_footprint.png")


In [ ]:
fig = survey.show()
fig.savefig(FIGS_OUT_DIR / "ddf_pointing_map.pdf")
fig.savefig(FIGS_OUT_DIR / "ddf_pointing_map.png")


## 6. Sky area covered by the DDF pointings

The DP2 visits are restricted to the Deep Drilling Fields, i.e. a handful of small, compact
patches of sky. We build the `skyarea` (a `shapely` polygon) as the convex hull of the observed
pointings, expanded by 1 deg, and use it to restrict the SN Ia draw to those DDF patches (with
`skysurvey.SNeIa.from_draw(skyarea=...)`) rather than the whole sky.

In [ ]:
from shapely.geometry import MultiPoint

points = MultiPoint(list(zip(survey_data["ra"], survey_data["dec"])))
skyarea = points.convex_hull.buffer(1.0)

print(f"DDF sky area (approx, incl. 1 deg buffer): {skyarea.area:.2f} deg^2 (planar approximation)")
skyarea


## 7. Draw the SN Ia population

In [ ]:
np.random.seed(RANDOM_SEED)

snia = skysurvey.SNeIa.from_draw(
    tstart=tmin, tstop=tmax,
    skyarea=skyarea,
    zmin=ZMIN, zmax=ZMAX,
)

print(f"{len(snia.data)} SNe Ia drawn (nature, before matching with the survey).")
snia.data.head()


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(snia.data["z"], bins=40, color="C0")
ax.set_xlabel("redshift")
ax.set_ylabel("N SNe Ia (generated)")
ax.set_title("Generated SN Ia redshift distribution (before survey matching)")
fig.tight_layout()
fig.savefig(FIGS_OUT_DIR / "sn_redshift_generated.pdf")
fig.savefig(FIGS_OUT_DIR / "sn_redshift_generated.png")


## 8. Match targets with the survey and build the light-curve dataset

`discard_bands=True` drops observations whose band falls outside the spectral range covered by
the SALT2/3 source model at the SN's redshift (see the
[Load the LSST survey](https://skysurvey.readthedocs.io/en/latest/howto/load_lsst.html) how-to for
details); this mostly removes some `y`-band points at low z and some `u`-band points at high z.

In [ ]:
dset = skysurvey.DataSet.from_targets_and_survey(
    snia, survey, progress_bar=True, discard_bands=True
)

ndet = dset.get_ndetection()
print(f"{len(ndet)} SNe Ia with at least one 5-sigma detection in the DP2 DDF visits "
      f"(out of {len(snia.data)} generated).")


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(ndet.values, bins=40, color="C1")
ax.set_xlabel("number of 5-sigma detections")
ax.set_ylabel("N SNe Ia")
ax.set_title("Detections per SN Ia in DP2 DDF")
fig.tight_layout()
fig.savefig(FIGS_OUT_DIR / "n_detections_per_sn.pdf")
fig.savefig(FIGS_OUT_DIR / "n_detections_per_sn.png")


In [ ]:
# Redshift distribution: generated vs. detected
detected_idx = ndet.index
fig, ax = plt.subplots(figsize=(5, 4))
bins = np.linspace(ZMIN, ZMAX, 40)
ax.hist(snia.data["z"], bins=bins, alpha=0.5, label="generated", color="C0")
ax.hist(snia.data.loc[detected_idx, "z"], bins=bins, alpha=0.7, label="detected (>=1 pt)", color="C1")
ax.set_xlabel("redshift")
ax.set_ylabel("N SNe Ia")
ax.legend()
ax.set_title("SN Ia redshift distribution: generated vs. DP2-DDF-detected")
fig.tight_layout()
fig.savefig(FIGS_OUT_DIR / "sn_redshift_generated_vs_detected.pdf")
fig.savefig(FIGS_OUT_DIR / "sn_redshift_generated_vs_detected.png")


In [ ]:
# Example light curve for the best-sampled SN Ia
example_index = ndet.idxmax()
fig = dset.show_target_lightcurve(index=example_index, phase_window=[-20, 40])
fig.savefig(FIGS_OUT_DIR / f"example_lightcurve_index{example_index}.pdf")
fig.savefig(FIGS_OUT_DIR / f"example_lightcurve_index{example_index}.png")
print(f"Example SN Ia index={example_index}, "
      f"z={snia.data.loc[example_index, 'z']:.3f}, "
      f"n_detections={ndet.loc[example_index]}")


## 9. Save data products

In [ ]:
snia.data.to_parquet(DATA_OUT_DIR / "snia_generated_catalog.parquet")
dset.data.to_parquet(DATA_OUT_DIR / "snia_dp2ddf_lightcurves.parquet")
ndet.to_frame("n_detections").to_parquet(DATA_OUT_DIR / "snia_ndetections.parquet")

print("Saved:")
for f in sorted(DATA_OUT_DIR.glob("*.parquet")):
    print(f"  - {f}")


## 10. Summary / next steps

- The SN Ia catalog matched to the DP2 DDF visits is in `dset.data`
  (`data_out_simSNIaDDF/snia_dp2ddf_lightcurves.parquet`).
- The generated (pre-matching) SN Ia population is in `snia.data`
  (`data_out_simSNIaDDF/snia_generated_catalog.parquet`).
- The adapted observing log (skysurvey format) is in
  `data_out_simSNIaDDF/dp2_ddf_survey_data.parquet` and can be reused directly with
  `skysurvey.LSST(data=...)` for other target classes (e.g. `skysurvey.SNeCC`).

**If Section 2's auto-detected column mapping was wrong or incomplete** for the real DP2 file,
edit `CANDIDATES` accordingly and re-run from Section 2 — everything downstream is driven by that
mapping. In particular, check:
- whether the visits table already contains a genuine `m5`/`fiveSigmaDepth`-like column (skip the
  SMTN-002 fallback in that case);
- whether `band` values need a different letter-extraction rule than `.str[-1]`;
- whether `airmass` is available directly, or needs to be derived from `zenithDistance`
  (`airmass = 1 / cos(zenithDistance)`).

Possible follow-ups: fit the simulated light curves with SALT2/SALT3
(consistent with your ELAsTiCC2 SALT2/SALT3 pipeline), compare the DP2-DDF SN Ia yield to the
full-LSST-footprint yield, or add Milky Way / host-galaxy dust effects
(see skysurvey's *Add Milky-Way extinction* how-to).
